# 15 — Global Alcohol Consumption: GroupBy Aggregations & The Legendary 'NA' Bug
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, SQL Aggregations, and Data Cleaning Interviews.*

---

## 📌 Executive Summary & Interview Expectations
This lab contains one of the **most famous data ingestion and aggregation interview traps in Pandas history**: the silent disappearance of an entire continent (**North America**) due to default `NaN` parsing.

### Core Competencies Tested in this Module:
1. **The 'NA' Ingestion Trap**: Why Pandas converts North America (`'NA'`) to missing values (`NaN`), dropping an entire continent from GroupBy operations by default!
2. **The `keep_default_na=False` Fix**: How to preserve legitimate two-letter country/continent codes.
3. **Numeric Aggregation Standards**: Why modern Pandas (2.x / 3.0) strictly requires `numeric_only=True` when applying statistical reductions to mixed-type DataFrames.
4. **Multi-Metric Groupby Summaries**: Extracting distribution statistics (`mean`, `min`, `max`, `describe()`) across geographical cohorts.
5. **Interview Corner**: The `dropna=False` parameter in `groupby()`, string-to-numeric reduction crashes, and country-level alcohol ratios.

## 1. Environment Setup & The Legendary 'NA' Ingestion Bug

### ⚠️ Top Interview Question: Where Did North America Go?
- In `drinks.csv`, continent codes are: `AS` (Asia), `EU` (Europe), `AF` (Africa), `SA` (South America), `OC` (Oceania), and **`NA` (North America)**.
- **The Trap**: By default, `pd.read_csv()` treats `'NA'`, `'N/A'`, `'null'`, and `'-nan'` as `NaN` (missing data)!
- **The Result**: 23 North American countries (USA, Canada, Mexico, etc.) silently become `NaN`. Because `df.groupby()` drops `NaN` keys by default, North America is **completely erased from all groupby calculations**!
- **The Fix**: Pass `na_values="", keep_default_na=False` to preserve literal `'NA'` strings.

In [1]:
import os
import numpy as np
import pandas as pd

# Load dataset URL fallback
csv_path = "drinks.csv"
if not os.path.exists(csv_path):
    csv_path = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/drinks.csv"

# ❌ The Naive Load (Demonstrating the Bug):
naive_drinks = pd.read_csv(csv_path)
print("Naive load unique continents:", list(naive_drinks["continent"].unique()))
print(f"Missing continents count: {naive_drinks['continent'].isna().sum()} (North America was erased!)")

# ✅ The Production Fix:
drinks = pd.read_csv(csv_path, na_values="", keep_default_na=False)
print("\nCorrected load unique continents:", list(drinks["continent"].unique()))
print(f"Missing continents count: {drinks['continent'].isna().sum()} (All 6 continents preserved!)")
drinks.head()

Naive load unique continents: ['AS', 'EU', 'AF', nan, 'SA', 'OC']
Missing continents count: 23 (North America was erased!)

Corrected load unique continents: ['AS', 'EU', 'AF', 'NA', 'SA', 'OC']
Missing continents count: 0 (All 6 continents preserved!)


,country,beer_servings,spirit_servings,wine_servings,total_litres_of_pure_alcohol,continent
0,Afghanistan,0,0,0,0.0,AS
1,Albania,89,132,54,4.9,EU
2,Algeria,25,0,14,0.7,AF
3,Andorra,245,138,312,12.4,EU
4,Angola,217,57,45,5.9,AF


## 2. Beer Consumption: Identifying the Leading Continent

In [2]:
# Average beer servings per continent
beer_by_continent = (
    drinks.groupby("continent")["beer_servings"]
    .mean()
    .sort_values(ascending=False)
)

print(f"🥇 Highest beer consuming continent: '{beer_by_continent.index[0]}' with {beer_by_continent.iloc[0]:.2f} servings on average.")
display(beer_by_continent)

🥇 Highest beer consuming continent: 'EU' with 193.78 servings on average.


continent
EU    193.777778
SA    175.083333
NA    145.434783
OC     89.687500
AF     61.471698
AS     37.045455
Name: beer_servings, dtype: float64

## 3. Wine Consumption: Five-Number Summaries by Continent

In [3]:
# Comprehensive distribution stats for wine per continent
drinks.groupby("continent")["wine_servings"].describe()

,count,mean,std,min,25%,50%,75%,max
continent,,,,,,,,
AF,53.0,16.264151,38.846419,0.0,1.0,2.0,13.00,233.0
AS,44.0,9.068182,21.667034,0.0,0.0,1.0,8.00,123.0
EU,45.0,142.222222,97.421738,0.0,59.0,128.0,195.00,370.0
NA,23.0,24.521739,28.266378,1.0,5.0,11.0,34.00,100.0
OC,16.0,35.625000,64.555790,0.0,1.0,8.5,23.25,212.0
SA,12.0,62.416667,88.620189,1.0,3.0,12.0,98.50,221.0


## 4. Mean & Median Consumption Across All Beverages

### ⚠️ Top Interview Error in Modern Pandas: `numeric_only=True`
- The `drinks` table contains a non-numeric column: `country` (`str`).
- In older Pandas, `drinks.groupby('continent').mean()` silently ignored `country`.
- In **Pandas 2.x and 3.0**, calling `.mean()` on non-numeric data raises:
  `TypeError: Could not convert ['country'] to numeric`.
- **Rule**: Always pass `numeric_only=True` when reducing DataFrames with mixed types!

In [4]:
# Mean consumption per continent across all numeric metrics
drinks.groupby("continent").mean(numeric_only=True).round(2)

,beer_servings,spirit_servings,wine_servings,total_litres_of_pure_alcohol
continent,,,,
AF,61.47,16.34,16.26,3.01
AS,37.05,60.84,9.07,2.17
EU,193.78,132.56,142.22,8.62
NA,145.43,165.74,24.52,6.00
OC,89.69,58.44,35.62,3.38
SA,175.08,114.75,62.42,6.31


In [5]:
# Median consumption per continent across all numeric metrics
drinks.groupby("continent").median(numeric_only=True)

,beer_servings,spirit_servings,wine_servings,total_litres_of_pure_alcohol
continent,,,,
AF,32.0,3.0,2.0,2.30
AS,17.5,16.0,1.0,1.20
EU,219.0,122.0,128.0,10.00
NA,143.0,137.0,11.0,6.30
OC,52.5,37.0,8.5,1.75
SA,162.5,108.5,12.0,6.85


## 5. Spirit Consumption: Multi-Metric Aggregation (`agg`)

In [6]:
# Mean, Min, and Max spirit consumption by continent
drinks.groupby("continent")["spirit_servings"].agg(
    ["mean", "min", "max"]
).round(2)

,mean,min,max
continent,,,
AF,16.34,0,152
AS,60.84,0,326
EU,132.56,0,373
NA,165.74,68,438
OC,58.44,0,254
SA,114.75,25,302


## 6. GroupBy & Data Ingestion Cheat Sheet

| Challenge / Operation | Idiomatic Syntax | Critical Interview Context |
| :--- | :--- | :--- |
| **Literal 'NA' Strings** | `pd.read_csv(..., keep_default_na=False)` | Prevents string 'NA' from becoming `NaN` |
| **Numeric Reduction** | `df.groupby('A').mean(numeric_only=True)` | Mandatory in Pandas 2.x/3.0 to prevent TypeError |
| **Preserve Null Groups** | `df.groupby('A', dropna=False)` | Groups by `NaN` as a valid category |
| **Targeted Series** | `df.groupby('A')['B'].mean()` | Avoids aggregating unwanted columns |
| **Multi-Metric Summary** | `df.groupby('A')['B'].agg(['min', 'mean', 'max'])` | Concise statistical profile |

---
## 🎯 7. Technical Interview Corner: Tricky Questions & Drills

### Q1: The `dropna=False` Parameter in GroupBy
**Question**: Suppose your dataset has records where the group key is `NaN` (e.g. unknown continent). By default, what does `df.groupby('continent').mean()` do with those rows? How do you include them?

**Answer**:
- By default, `df.groupby()` **drops missing values (`NaN`) from the group index**! Those rows are completely ignored and omitted from output summaries.
- In Pandas 1.1+, pass **`dropna=False`** in `.groupby()`:
  `df.groupby('continent', dropna=False).mean(numeric_only=True)`.
  This creates an explicit group labeled `NaN` so missing keys are audited rather than hidden!

In [7]:
# Demonstration of dropna=False in groupby
demo_df = pd.DataFrame({
    "Category": ["Electronics", "Clothing", None, "Electronics", None],
    "Revenue": [500, 200, 150, 400, 300]
})

print("Default groupby (drops null group):")
display(demo_df.groupby("Category")["Revenue"].sum())

print("\nGroupby with dropna=False (preserves null group):")
display(demo_df.groupby("Category", dropna=False)["Revenue"].sum())

Default groupby (drops null group):


Category
Clothing       200
Electronics    900
Name: Revenue, dtype: int64


Groupby with dropna=False (preserves null group):


Category
Clothing       200
Electronics    900
NaN            450
Name: Revenue, dtype: int64

### Q2: Advanced Interview Challenge: Beer Dominance Index
**Challenge**: In a single chained expression, find the top 5 countries in the world where **beer represents the highest percentage of total alcohol servings**, considering only countries with **at least 100 total servings** of alcohol!

In [8]:
# Solution to Coding Challenge
beer_dominant = (
    drinks.assign(
        total_servings=lambda df: df["beer_servings"] + df["spirit_servings"] + df["wine_servings"],
        beer_share_pct=lambda df: (df["beer_servings"] / df["total_servings"]) * 100
    )
    .query("total_servings >= 100")
    .sort_values(by="beer_share_pct", ascending=False)
    [["country", "continent", "beer_servings", "total_servings", "beer_share_pct"]]
    .head(5)
    .round(1)
    .reset_index(drop=True)
)

print("Top 5 Beer-Dominant Countries (min 100 total servings):")
display(beer_dominant)

Top 5 Beer-Dominant Countries (min 100 total servings):


,country,continent,beer_servings,total_servings,beer_share_pct
0,Namibia,AF,376,380,98.9
1,Vietnam,AS,111,114,97.4
2,Cameroon,AF,147,152,96.7
3,South Korea,AS,140,165,84.8
4,Palau,OC,306,392,78.1
